# 08. Segmentation for Optical Imaging: U-Net, Losses, Metrics, and QC

**Author:** Md. Mobarak Karim, Ph.D.  
**Level:** Beginner → research-practical  
**Course:** Deep Learning for Optical Imaging

This notebook builds a research-quality segmentation baseline and explains the choices behind it.

> **How to study this notebook:** read the explanation first, predict what the code should do, run it, change one parameter, and explain why the result changed.


## Learning objectives

- Explain U-Net conceptually
- Design binary/multi-class outputs
- Implement Dice correctly
- Combine BCE and Dice thoughtfully
- Select thresholds correctly
- Recognize thin/small-structure failure modes
- Validate using overlays and per-specimen metrics


## Mind map

```mermaid
mindmap
  root((Segmentation))
    Model
      Encoder
      Decoder
      Skip connections
    Output
      Binary
      Multiclass
    Loss
      BCE
      Dice
      Combined
    Metrics
      Dice
      IoU
      Sensitivity
      Specificity
    Optical QC
      Thin structures
      Boundaries
      Dim signal
      Artifacts

```


## 1. Why U-Net is a strong first segmentation baseline

U-Net combines:
- an **encoder** that extracts increasingly abstract features;
- a **decoder** that restores spatial resolution;
- **skip connections** that bring fine spatial detail from encoder to decoder.

For optical imaging, this is useful because segmentation often needs both:
- context ("this region belongs to a vessel/layer/cell"), and
- precise boundaries.


## 2. Output design

For binary segmentation:
- model output: one logit per pixel;
- target: 0/1 mask;
- common loss: `BCEWithLogitsLoss` + Dice loss;
- convert logits to probability using sigmoid only for interpretation/inference.

For multi-class segmentation:
- model output: `C` logits per pixel;
- target: integer class index per pixel;
- common loss: `CrossEntropyLoss` + optional Dice variant.


## 3. Dice coefficient

For binary masks:

```text
Dice = 2 * |prediction ∩ target| / (|prediction| + |target|)
```

It emphasizes overlap and is useful when foreground occupies a small fraction of the image.

### Caution
Dice can hide clinically/scientifically important boundary errors. Always inspect overlays and consider class-specific sensitivity/boundary metrics when relevant.


In [ ]:
import torch

def dice_score_from_binary(pred, target, eps=1e-8):
    pred = pred.float()
    target = target.float()
    inter = (pred * target).sum(dim=(-2, -1))
    denom = pred.sum(dim=(-2,-1)) + target.sum(dim=(-2,-1))
    return ((2*inter + eps)/(denom + eps)).mean()

target = torch.tensor([[[[1,1],[0,0]]]], dtype=torch.float32)
pred_good = target.clone()
pred_bad = 1-target

print("good:", dice_score_from_binary(pred_good, target).item())
print("bad :", dice_score_from_binary(pred_bad, target).item())


## 4. Dice loss from probabilities

A differentiable Dice-style loss uses soft probabilities rather than thresholded masks.

Do **not** threshold inside the training loss; thresholding is nondifferentiable.


In [ ]:
import torch

def soft_dice_loss(logits, target, eps=1e-6):
    prob = torch.sigmoid(logits)
    target = target.float()
    dims = (-2, -1)
    inter = (prob * target).sum(dim=dims)
    denom = prob.sum(dim=dims) + target.sum(dim=dims)
    dice = (2*inter + eps)/(denom + eps)
    return 1 - dice.mean()

logits = torch.randn(2,1,64,64)
target = torch.randint(0,2,(2,1,64,64)).float()
print(soft_dice_loss(logits, target))


## 5. BCE + Dice: why combine them?

BCE provides per-pixel supervision; Dice emphasizes region overlap.

A common starting point:

```text
L_total = lambda_bce * BCE + lambda_dice * DiceLoss
```

But the coefficients are hyperparameters. Inspect the magnitude of each loss term before combining them so one term does not dominate unintentionally.


## 6. Threshold selection

A default threshold of 0.5 is common, but it is not sacred.

If threshold is tuned, tune it on the **validation set**, then freeze it before test evaluation.

For highly imbalanced problems, report sensitivity/specificity versus threshold or precision-recall behavior when useful.


## 7. Small-object and boundary issues in optical imaging

Examples:
- dim microvessels may be missed;
- thin retinal layers may collapse after downsampling;
- tiny nuclei can merge;
- bright artifacts can become false positives.

Possible responses:
- preserve resolution;
- use patch size with adequate context;
- class/boundary-aware losses;
- hard-example sampling;
- postprocessing only if scientifically justified and documented.

Do not hide poor model behavior with aggressive morphology.


## 8. Validation overlays

Always compare:
- raw image;
- ground-truth mask;
- predicted probability;
- thresholded mask;
- overlay of errors.

Look specifically at false negatives and false positives in scientifically meaningful structures.


## 9. Segmentation checklist for optical imaging

```text
□ Independent specimen split
□ Label definition documented
□ Annotation uncertainty considered
□ Output classes exactly match target encoding
□ Tiny-set overfit succeeds
□ BCE/Dice implementation tested
□ Threshold tuned only on validation
□ Per-specimen metrics saved
□ Representative and worst-case overlays inspected
□ External acquisition/site tested when possible
```


## End-of-notebook checklist

Before moving on, you should be able to explain the main ideas **without looking at the code**. If you cannot explain why a method, loss, split, or metric is appropriate, repeat the relevant section before using it in research.
